# Data Processing

<br><br>

## 0. Imports

In [1]:
import numpy as np
import pandas as pd

<br><br>

# 1. Load Data

In [2]:
diabetes_raw = pd.read_csv('data/diabetes/diabetic_data.csv')

<br><br>

## 2. Clean Data

In [3]:
# Drop unknown gender
diabetes_clean = diabetes_raw[diabetes_raw['gender'] != 'Unknown/Invalid']

diabetes_clean['race'] = diabetes_clean['race'].fillna('Unknown')
diabetes_clean['payer_code'] = diabetes_clean['payer_code'].fillna('Unknown')
diabetes_clean['medical_specialty'] = diabetes_clean['medical_specialty'].fillna('Unknown')
diabetes_clean['max_glu_serum'] = diabetes_clean['max_glu_serum'].fillna('None')
diabetes_clean['A1Cresult'] = diabetes_clean['A1Cresult'].fillna('None')

# Drop high-missing columns
diabetes_clean = diabetes_clean.drop(columns = 'weight')

# Drop patients who cannot be readmitted
diabetes_clean = diabetes_clean[
    ~diabetes_clean['discharge_disposition_id'].isin({11, 13, 14, 19, 20, 21})
]

# Clean columns
diabetes_clean = diabetes_clean.replace({'?': np.nan})
diabetes_clean['readmitted'] = diabetes_clean['readmitted'].replace({'NO': 0,'>30': 0,'<30': 1}).astype(int)
diabetes_clean['diabetesMed'] = diabetes_clean['diabetesMed'].replace({'No': 0, 'Yes': 1}).astype(int)
diabetes_clean['change'] = diabetes_clean['change'].replace({'No': 0, 'Ch': 1}).astype(int)
diabetes_clean['gender'] = diabetes_clean['gender'].replace({'Male': 1, 'Female': 0}).astype(int)

# Keep only the first record for each patient_nbr
diabetes_clean = diabetes_clean.sort_values('encounter_id')
diabetes_clean = diabetes_clean.groupby('patient_nbr').first().reset_index()

# Drop identifiers
diabetes_clean = diabetes_clean.drop(columns=['encounter_id', 'patient_nbr'])

In [4]:
diabetes_clean.columns

Index(['race', 'gender', 'age', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'payer_code', 'medical_specialty', 'num_lab_procedures',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3',
       'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin',
       'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
       'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')

<br><br>

## 3. Process Data

In [5]:
diabetes_processed = diabetes_clean.copy()

### 3.1. Transform Age

In [6]:
# Map age groups to numeric values
diabetes_processed['age'] = diabetes_processed['age'].map({
    '[0-10)': 5,
    '[10-20)': 15,
    '[20-30)': 25,
    '[30-40)': 35,
    '[40-50)': 45,
    '[50-60)': 55,
    '[60-70)': 65,
    '[70-80)': 75,
    '[80-90)': 85,
    '[90-100)': 95
})

### 3.2. Transform Medicin

In [7]:
medication_cols = [
    'metformin',
    'repaglinide',
    'nateglinide',
    'chlorpropamide',
    'glimepiride',
    'acetohexamide',
    'glipizide',
    'glyburide',
    'tolbutamide',
    'pioglitazone',
    'rosiglitazone',
    'acarbose',
    'miglitol',
    'troglitazone',
    'tolazamide',
    'examide',
    'citoglipton',
    'insulin',
    'glyburide-metformin',
    'glipizide-metformin',
    'glimepiride-pioglitazone',
    'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

for col in medication_cols:
    diabetes_processed[col] = diabetes_processed[col].apply(
        lambda x: 0 if x == 'No' else 1
    )

In [8]:
rare_meds = [
    'acarbose',
    'chlorpropamide',
    'tolazamide',
    'miglitol',
    'tolbutamide',
    'glipizide-metformin',
    'troglitazone',
    'metformin-rosiglitazone',
    'acetohexamide',
    'glimepiride-pioglitazone',
    'metformin-pioglitazone',
    'examide',
    'citoglipton'
]

diabetes_processed["other_meds"] = (
    diabetes_processed[rare_meds].sum(axis=1) > 0
).astype(int)

diabetes_processed = diabetes_processed.drop(columns=rare_meds)

### 3.3. Transform Diagnoses

In [9]:
def categorize_diagnosis(diag):
    if pd.isna(diag):
        return "Other"

    diag = str(diag).strip()

    # E-codes and V-codes → Other (paper groups these in Other)
    if diag.startswith("E") or diag.startswith("V"):
        return "Other"

    try:
        diag_num = float(diag)

        # Diabetes: 250.xx
        if 250 <= diag_num < 251:
            return "Diabetes"

        # Neoplasms: 140–239
        if 140 <= diag_num < 240:
            return "Neoplasms"

        # Circulatory: 390–459, 785
        if (390 <= diag_num < 460) or (785 <= diag_num < 786):
            return "Circulatory"

        # Respiratory: 460–519, 786
        if (460 <= diag_num < 520) or (786 <= diag_num < 787):
            return "Respiratory"

        # Digestive: 520–579, 787
        if (520 <= diag_num < 580) or (787 <= diag_num < 788):
            return "Digestive"

        # Genitourinary: 580–629, 788
        if (580 <= diag_num < 630) or (788 <= diag_num < 789):
            return "Genitourinary"

        # Musculoskeletal: 710–739
        if 710 <= diag_num < 740:
            return "Musculoskeletal"

        # Injury: 800–999
        if 800 <= diag_num <= 999:
            return "Injury"

        return "Other"

    except:
        return "Other"

diabetes_processed["diag_1_group"] = diabetes_processed["diag_1"].apply(categorize_diagnosis)
diabetes_processed["diag_2_group"] = diabetes_processed["diag_2"].apply(categorize_diagnosis)
diabetes_processed["diag_3_group"] = diabetes_processed["diag_3"].apply(categorize_diagnosis)

# List all diagnosis group columns
diag_group_cols = [
    "diag_1_group",
    "diag_2_group",
    "diag_3_group"
]

# Get all unique disease categories
all_categories = pd.unique(
    diabetes_processed[diag_group_cols].values.ravel()
)

# Create binary columns
for category in all_categories:
    diabetes_processed[category] = (
        (diabetes_processed["diag_1_group"] == category) |
        (diabetes_processed["diag_2_group"] == category) |
        (diabetes_processed["diag_3_group"] == category)
    ).astype(int)

diabetes_processed = diabetes_processed.drop(columns=[
    "diag_1",
    "diag_2",
    "diag_3",
    "diag_1_group",
    "diag_2_group",
    "diag_3_group",
])

### 3.4. Combine Visit Counts

In [10]:
diabetes_processed['service_utilization'] = (
    diabetes_processed['number_outpatient'] + 
    diabetes_processed['number_emergency'] + 
    diabetes_processed['number_inpatient']
)

# Drop the individual counts
diabetes_processed = diabetes_processed.drop(columns=['number_outpatient', 'number_emergency', 'number_inpatient'])

In [11]:
diabetes_processed.info()

<class 'pandas.DataFrame'>
RangeIndex: 69987 entries, 0 to 69986
Data columns (total 39 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   race                      68166 non-null  str  
 1   gender                    69987 non-null  int64
 2   age                       69987 non-null  int64
 3   admission_type_id         69987 non-null  int64
 4   discharge_disposition_id  69987 non-null  int64
 5   admission_source_id       69987 non-null  int64
 6   time_in_hospital          69987 non-null  int64
 7   payer_code                40591 non-null  str  
 8   medical_specialty         37934 non-null  str  
 9   num_lab_procedures        69987 non-null  int64
 10  num_procedures            69987 non-null  int64
 11  num_medications           69987 non-null  int64
 12  number_diagnoses          69987 non-null  int64
 13  max_glu_serum             69987 non-null  str  
 14  A1Cresult                 69987 non-null  str  
 

### 3.5. One-Hot Encode Remaining Categorical Columns

In [12]:
# Reduce cardinality for medical_specialty
top_specialties = diabetes_processed['medical_specialty'].value_counts().nlargest(10).index
diabetes_processed['medical_specialty'] = diabetes_processed['medical_specialty'].apply(
    lambda x: x if x in top_specialties else 'Other'
)

# Group rare payer codes (n < 100) into 'Other'
rare_payers = diabetes_processed['payer_code'].value_counts()
rare_payers = rare_payers[rare_payers < 100].index
diabetes_processed['payer_code'] = diabetes_processed['payer_code'].apply(
    lambda x: 'Other' if x in rare_payers else x
)

# Make "Caucasian" the base race category (helpful for SHAP later)
diabetes_processed['race'] = pd.Categorical(
    diabetes_processed['race'],
    categories = ['Caucasian', 'AfricanAmerican', 'Asian', 'Hispanic', 'Other', 'Unknown']
)

# Encode
categorical_cols = ['race', 'payer_code', 'medical_specialty', 'max_glu_serum', 'A1Cresult']
diabetes_processed = pd.get_dummies(
    diabetes_processed, 
    columns=categorical_cols, 
    drop_first=True,
    dtype=int
)

In [13]:
def bin_admission_type(val):
    if val in [1, 2, 7]: return 'Emergency'
    if val == 3: return 'Elective'
    return 'Other'

def bin_discharge_disposition(val):
    if val == 1: return 'Home'
    if val in [3, 6, 8]: return 'Home_Health_SNF'
    if val in [2, 4, 5, 22, 23, 24, 27, 28, 29, 30]: return 'Transferred_Medical'
    return 'Other'

def bin_admission_source(val):
    if val == 7: return 'Emergency_Room'
    if val in [1, 2, 3]: return 'Referral'
    if val in [4, 5, 6, 10, 18, 22, 25, 26]: return 'Transfer'
    return 'Other'

# Apply transformations
diabetes_processed['admission_type'] = diabetes_processed['admission_type_id'].apply(bin_admission_type)
diabetes_processed['discharge_disposition'] = diabetes_processed['discharge_disposition_id'].apply(bin_discharge_disposition)
diabetes_processed['admission_source'] = diabetes_processed['admission_source_id'].apply(bin_admission_source)

# Drop original numerical ID columns
diabetes_processed = diabetes_processed.drop(columns=['admission_type_id', 'discharge_disposition_id', 'admission_source_id'])

# One-hot encode new categorical columns
categorical_to_encode = ['admission_type', 'discharge_disposition', 'admission_source']
diabetes_processed = pd.get_dummies(diabetes_processed, columns=categorical_to_encode, drop_first=True, dtype=int)

In [14]:
diabetes_processed.info()

<class 'pandas.DataFrame'>
RangeIndex: 69987 entries, 0 to 69986
Data columns (total 73 columns):
 #   Column                                        Non-Null Count  Dtype
---  ------                                        --------------  -----
 0   gender                                        69987 non-null  int64
 1   age                                           69987 non-null  int64
 2   time_in_hospital                              69987 non-null  int64
 3   num_lab_procedures                            69987 non-null  int64
 4   num_procedures                                69987 non-null  int64
 5   num_medications                               69987 non-null  int64
 6   number_diagnoses                              69987 non-null  int64
 7   metformin                                     69987 non-null  int64
 8   repaglinide                                   69987 non-null  int64
 9   nateglinide                                   69987 non-null  int64
 10  glimepiride          

<br><br>

## 4. Save Processed Data

In [15]:
diabetes_processed.to_csv('data/diabetes_processed.csv', index = False)